---
## 0. Imports

In [ ]:
import numpy as np
import os
import rasterio
from rasterio.windows import from_bounds
from pystac_client import Client
from pyproj import Transformer
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'EMPTY_DIR'
os.environ['CPL_VSIL_CURL_ALLOWED_EXTENSIONS'] = '.tif,.TIF'
os.environ['GDAL_HTTP_MERGE_CONSECUTIVE_RANGES'] = 'YES'

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

print("Imports OK.")

---
## 1. Configuration

In [ ]:
DATE_START = "2024-01-01"
DATE_END   = "2024-03-31"

MAX_CLOUD_COVER = 20        # percent
MAX_SCENES = 5              # max scenes to process (set higher if needed)

BBOX = [-46.0, -23.5, -45.5, -23.0]   # [west, south, east, north] in EPSG:4326

OUTPUT_DIR = "../output/sentinel2_multi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Date range:      {DATE_START} to {DATE_END}")
print(f"Cloud cover:     ≤ {MAX_CLOUD_COVER}%")
print(f"Max scenes:      {MAX_SCENES}")
print(f"Bounding box:    {BBOX}")

---
## 2. Search Sentinel-2 L2A

In [ ]:
catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=f"{DATE_START}/{DATE_END}",
    query={"eo:cloud_cover": {"lt": MAX_CLOUD_COVER}},
    max_items=100,
)

all_items = list(search.items())
print(f"Found {len(all_items)} scenes with cloud cover < {MAX_CLOUD_COVER}%")

# Sort by date
all_items.sort(key=lambda x: x.datetime)

print(f"\n{'#':>3}  {'Date':12s}  {'Cloud%':>7s}  {'Tile':6s}  {'ID'}")
print("-" * 80)
for i, item in enumerate(all_items):
    dt = item.datetime.strftime("%Y-%m-%d") if item.datetime else "N/A"
    cloud = item.properties.get("eo:cloud_cover", -1)
    tile = item.properties.get("s2:mgrs_tile", "?")
    marker = " ◄" if i < MAX_SCENES else ""
    print(f"{i:3d}  {dt:12s}  {cloud:6.1f}%  {tile:6s}  {item.id}{marker}")

### 2.1 Select scenes to process

By default, takes the first `MAX_SCENES`.

In [ ]:
# =====================================================================
#  Option A: Take the first MAX_SCENES (default)
selected_items = all_items[:MAX_SCENES]

#  Option B: Pick specific indices from the table above
#  selected_items = [all_items[0], all_items[3], all_items[7]]
# =====================================================================

print(f"Selected {len(selected_items)} scenes for processing:")
for item in selected_items:
    print(f"  {item.datetime.strftime('%Y-%m-%d')}  {item.id}")

---
## 3. Helper Function

In [ ]:
def read_band_in_bbox(item, asset_key, bbox_4326):
    """Read a single band from a STAC item, clipped to a BBOX."""
    url = item.assets[asset_key].href
    west, south, east, north = bbox_4326
    with rasterio.open(url) as src:
        tr = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        img_left, img_bottom = tr.transform(west, south)
        img_right, img_top = tr.transform(east, north)
        window = from_bounds(img_left, img_bottom, img_right, img_top, src.transform)
        data = src.read(1, window=window)
        profile = src.profile.copy()
        profile.update(width=data.shape[1], height=data.shape[0],
                       transform=src.window_transform(window))
    return data, profile

print("Helper defined.")

---
## 4. Process All Selected Scenes

In [ ]:
s2_results = {}  # keyed by date string

for item in selected_items:
    date_str = item.datetime.strftime("%Y%m%d")
    label = f"s2_{date_str}"
    print(f"\nProcessing {item.id} ...")

    # Read SCL
    try:
        scl, scl_profile = read_band_in_bbox(item, "scl", BBOX)
    except Exception as e:
        print(f"  ERROR reading SCL: {e}. Skipping.")
        continue

    # Extract water mask
    scl_water = (scl == 6).astype(np.uint8)
    scl_water[scl == 0] = 255  # NoData

    # Save
    out_path = os.path.join(OUTPUT_DIR, f"s2_scl_water_{date_str}.tif")
    out_profile = scl_profile.copy()
    out_profile.update(dtype=rasterio.uint8, count=1, nodata=255, compress="lzw")
    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(scl_water, 1)

    n_water = (scl_water == 1).sum()
    n_valid = (scl_water != 255).sum()
    print(f"  Shape: {scl.shape}, Water: {n_water:,d} px ({100*n_water/max(n_valid,1):.2f}%)")
    print(f"  Saved: {out_path}")

    s2_results[label] = {
        "path": out_path,
        "mask": scl_water,
        "date": date_str,
        "item_id": item.id,
    }

print(f"\n\nProcessed {len(s2_results)} Sentinel-2 scenes.")

---
## 5. Visual Summary

In [ ]:
n = len(s2_results)
if n == 0:
    print("No scenes processed.")
else:
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    for i, (label, r) in enumerate(s2_results.items()):
        row, col = divmod(i, cols)
        ax = axes[row, col]
        display = r["mask"].astype(float)
        display[r["mask"] == 255] = np.nan
        ax.imshow(display, cmap="Blues", vmin=0, vmax=1)
        n_w = (r["mask"] == 1).sum()
        ax.set_title(f"{r['date']}\n{n_w:,d} water px", fontsize=9)
        ax.axis("off")

    # Hide unused axes
    for i in range(n, rows * cols):
        row, col = divmod(i, cols)
        axes[row, col].axis("off")

    fig.suptitle("Sentinel-2 SCL Water Masks — All Scenes", fontsize=13, y=1.01)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "summary_all_scenes.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
print("Output files:")
for label, r in s2_results.items():
    print(f"  {r['date']}  →  {r['path']}")
print("\nDone.")